### eNextSUT generator

Run this code to generate (or update) our eNextSUT reference database, reliying on Exiobase Hybrid version (v3.3.18). 

The above-mentioned database is parsed, aggregated in terms of electricity commodity (1 commodity) and activities (matching the EMBER power plants technologies resolution and labelling) and electricity production mixes are updated according to EMBER ones for a given year.

Just mind to update your paths in the 'paths.yml' file and to specify the user (your initials) and the desired electricity mix in the first cell, then run all the code

---
##### Update from 2026!



---


In [7]:
import mario
import yaml
import pandas as pd
import os

from support.ember_remapping import map_ember_to_classification
import warnings
warnings.filterwarnings("ignore")

user = 'LR'   # change this to your username
year = 2024   # change this to the year you want to update the electricity mixes to

with open('paths.yml', 'r') as file: # open the yml file
    paths = yaml.safe_load(file)

paths = paths[user]

Parse raw Exiobase database

In [8]:
db = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')

Aggregating electricity commodities and activities to match EMBER resolution


In [9]:
db.aggregate("support/aggregate_ee.xlsx",ignore_nan=True)

nan values for the aggregation of Activity for following items ignored
['Cultivation of paddy rice', 'Cultivation of wheat', 'Cultivation of cereal grains nec', 'Cultivation of vegetables, fruit, nuts', 'Cultivation of oil seeds', 'Cultivation of sugar cane, sugar beet', 'Cultivation of plant-based fibers', 'Cultivation of crops nec', 'Cattle farming', 'Pigs farming', 'Poultry farming', 'Meat animals nec', 'Animal products nec', 'Raw milk', 'Wool, silk-worm cocoons', 'Manure treatment (conventional), storage and land application', 'Manure treatment (biogas), storage and land application', 'Forestry, logging and related service activities (02)', 'Fishing, operating of fish hatcheries and fish farms; service activities incidental to fishing (05)', 'Mining of coal and lignite; extraction of peat (10)', 'Extraction of crude petroleum and services related to crude oil extraction, excluding surveying', 'Extraction of natural gas and services related to natural gas extraction, excluding surve

In [10]:
# Parse ember electricity generation data, map to exiobase and get electricity mix for a given year 
ee_mix = map_ember_to_classification(
    path = paths['ember'],
    classification = 'EXIO3',
    year = None,
    mode = 'mix',
)

Updating electricity supply mixes

In [11]:
z = db.z
s = db.s

for region in db.get_index('Region'):
    print(region,end=' ')
    region_latest_year = ee_mix.loc[(region,slice(None),slice(None))].index.get_level_values(0).max()
    mix_year = year if year <= region_latest_year else region_latest_year
    
    new_mix = ee_mix.loc[(region,mix_year,slice(None)),'Value'].to_frame().sort_index(axis=0)
    new_mix.index = new_mix.index.get_level_values(2)
    
    old_market_share = s.loc[(region, 'Activity', new_mix.index),(region,'Commodity','Electricity')].sum().sum()
    
    s.loc[(region, 'Activity', new_mix.index),(region,'Commodity','Electricity')] = new_mix.values*old_market_share # check if commodity electricity is called "Electricity" in aggregation excel file
    print('done')

z.update(s)

db.update_scenarios('baseline',z=z)
db.reset_to_coefficients('baseline')

FI done
FR done
LU done
IE done
LV done
US done
WA done
WE done
DK done
KR done
PT done
EE done
BE done
RU done
NO done
DE done
WM done
CN done
CY done
CZ done
TR done
AU done
ID done
IT done
SK done
CA done
WF done
HR done
MT done
ES done
BR done
AT done
BG done
NL done
RO done
GB done
ZA done
WL done
MX done
SI done
GR done
CH done
LT done
SE done
IN done
PL done
JP done
HU done


Export the v1.0 database

In [12]:
db.to_txt(
    path = os.path.join(
        paths['export'],
        "v1.0",
        str(year),
        ),
    )

Database: to calculate V following matrices are need.
['X'].Trying to calculate dependencies.


---
#### Continue to get v2.0 database
Define a list of commodities whose trade mixes are to be updated is defined (e.g. Electricity) and add related sectors as empty rows and cols in the table


In [13]:
traded_commodities = ['Electricity']

for commodity in traded_commodities:
    new_activities = [f"{commodity} supply"]
    new_commdodities = [f"{commodity} need"]

db.add_sectors(
    new_sectors = new_activities,
    regions = db.get_index("Region"),
    io ="support/add_sectors_activities.xlsx",
    item = "Activity",
    inplace = True,
)

db.add_sectors(
    new_sectors = new_commdodities,
    regions = db.get_index("Region"),
    io ="support/add_sectors_commodities.xlsx",
    item = "Commodity",
    inplace = True,
)

Database: All the scenarios will be deleted from the database
Database: All the scenarios will be deleted from the database



### Demand-side shock
1. Activities supplying such new commodities must consume only domestic "original" commodity (e.g. "Electricity supply" activity must only consume domestic "Electricity" commodity)
2. The consumption of the original commodity (both domestic and imported) must be transferred to domestic "Need" commodity consumption (both for use and final demand)


In [14]:
u_new = db.u.copy()
Y_new = db.Y.copy()

U = db.U.copy().loc[(slice(None),"Commodity",traded_commodities),:].groupby(level=[0],axis=1).sum()
Y = Y_new.loc[(slice(None),"Commodity",traded_commodities),:].groupby(level=[0],axis=1).sum()
UY = U + Y

z_new = db.z.copy()

trades_df = {}

for commodity in traded_commodities:
    trades_df[commodity] = pd.DataFrame()
    u_new.loc[:,(slice(None),"Activity",f"{commodity} supply")] *= 0
    oth_activities = [i for i in db.get_index("Activity") if i != f"{commodity} supply"]
    
    for region in db.get_index("Region"):
        u_new.loc[(region,"Commodity",commodity),(region,"Activity",f"{commodity} supply")] = 1

        ee_consumption_u = db.u.loc[(slice(None),"Commodity",commodity),(region,"Activity",oth_activities)].sum(0).to_frame().T
        ee_consumption_u.index = pd.MultiIndex.from_arrays([[region],["Commodity"],[f"{commodity} need"]],names=db.u.index.names)

        ee_consumption_Y = db.Y.loc[(slice(None),"Commodity",commodity),(region,"Consumption category",slice(None))].sum(0).to_frame().T
        ee_consumption_Y.index = pd.MultiIndex.from_arrays([[region],["Commodity"],[f"{commodity} need"]],names=db.Y.index.names)

        u_new.update(ee_consumption_u)
        Y_new.update(ee_consumption_Y)

        u_new.loc[(slice(None),"Commodity",commodity),(region,"Activity",oth_activities)] *= 0
        Y_new.loc[(slice(None),"Commodity",commodity),(region,"Consumption category",slice(None))] *= 0

        trades_df[commodity] = pd.concat([
            trades_df[commodity], 
            UY.loc[:,region]/UY.loc[:,region].sum()
        ], axis=1
        )

z_new.update(u_new)


Update baseline scenario and reset database to coefficients


In [15]:
db.update_scenarios(scenario='baseline',z=z_new, Y=Y_new)
db.reset_to_coefficients('baseline')

### Supply-side shock
1. Get shock template to be filled with data on commodity trades (e.g. manually scrapping data from EE maps). "Supply" activities must provide "need" commodity according to trades dataframe
2. Use the shock_calc function to apply the shock on trades mixes

In [16]:
# db.get_shock_excel("support/trades.xlsx"))
db.shock_calc("support/trades.xlsx", z=True, scenario='ee_trades', force_rewrite=True)

Database: to calculate Z following matrices are need.
['X'].Trying to calculate dependencies.


Export the v2.0 database


In [17]:
#%% Export the v2.0 database
db.to_txt(
    path = os.path.join(
        paths['export'],
        "v2.0",
        str(year),
        ),
    scenario = 'ee_trades',
    # flows=True,
    # coefficients=True
    )
